## Setup

In [26]:
# If you get import errors, uncomment the next two lines:
#!pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
#!pip install --quiet pillow

import torch, torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image, ImageFile
import torch.nn.functional as F

device = torch.device("mps")

## Load data

In [27]:
# Use the data folder "1_Case_AEC", which contains a subset of images from: 
# https://www.kaggle.com/datasets/shaunthesheep/microsoft-catsvsdogs-dataset 
# (Downloading and unzipping might take some time; we require many images to train the classifier)

# Change this path to where you unzipped the Kaggle dataset so that:
# DATA_DIR/
#   ├─ Cat/
#   └─ Dog/

DATA_DIR = "data/case1/"  # <-- EDIT ME!


# ImageFolder expects subfolders per class (Cat, Dog)
ImageFile.LOAD_TRUNCATED_IMAGES = True  # tolerate slightly broken JPEGs

# Return True only if `path` is a real, readable image.
#    What this does:
#    - Tries to open the file with Pillow (PIL) without fully decoding the pixels.
#    - `verify()` performs a quick integrity check (headers, structure).
#    - If the file is empty, corrupted, or not an image, Pillow raises an exception.
#    - We catch that and return False so bad files are filtered out safely.


def ok(path):
    # return True only if the file opens as an image (very short, very robust)
    try:
        Image.open(path).verify()
        return True
    except:
        return False

## Preprocessing

In [28]:
# Use the same preprocessing that the pretrained weights expect (easy & reliable)
weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

full_ds = ImageFolder(root=DATA_DIR, transform=preprocess, is_valid_file=ok)
print("Valid images:", len(full_ds), "| classes:", full_ds.classes)

/Users/jan.vaorin/PycharmProjects/JupyterProject1/.venv/lib/python3.13/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Valid images: 12498 | classes: ['Cat', 'Dog']


## DataLoader; Train/Test split

In [37]:
from torch.utils.data import random_split, DataLoader


# Split dataset: 80% training, 20% testing
train_size = int(0.95 * len(full_ds))
test_size = len(full_ds) - train_size

train_ds, test_ds = random_split(full_ds, [train_size, test_size])

print(f"Training samples: {len(train_ds)}")
print(f"Test samples: {len(test_ds)}")

# Create DataLoaders
BATCH = 32

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH,
    shuffle=True  # Randomize order each epoch
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH,
    shuffle=False  # Keep consistent for evaluation
)

print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

Training samples: 11873
Test samples: 625
Training batches: 372
Test batches: 20


## Build a small model (e.g., freeze ResNet-18, train only the last layer)

In [46]:
# Pretrained backbone => We are building on an existing model with trained weights, this process is called transfer learning!
model = resnet18(weights=weights)

# Freeze all pretrained layers for speed/simplicity
for p in model.parameters():
    p.requires_grad = False

# Replace the final layer with a 2-class classifier (Cat/Dog)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = torch.nn.CrossEntropyLoss()

# Select an optimizer below
# Then, we only pass model.fc.parameters() because we *froze* the pretrained backbone
# and want to train just the final layer. lr=1e-3 is a safe starting learning rate.

#optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)


## Training

In [47]:
# This might take several minutes due to complex/thorough training and the large number of images (>12.000)
model.train()  # put the model in *training mode* (enables layers like dropout/batchnorm to update)

# counters to track total loss and accuracy over the whole epoch
running_loss, total, correct = 0.0, 0, 0

for imgs, labels in train_loader:          # iterate over mini-batches of training data
    imgs, labels = imgs.to(device), labels.to(device)  # move data to GPU if available (else CPU)

    optimizer.zero_grad()                  # clear old gradients from the previous step
    logits = model(imgs)                   # forward pass: raw class scores for each image
    loss = criterion(logits, labels)       # compute how far predictions are from true labels

    loss.backward()                        # backpropagate: compute gradients
    optimizer.step()                       # update trainable parameters using the gradients

    # accumulate *sum* of losses so we can compute an average later
    running_loss += loss.item() * imgs.size(0)  # loss per sample × batch size

    # turn raw scores into predicted class indices (the highest score wins)
    preds = logits.argmax(1)

    # count how many predictions were correct in this batch
    correct += (preds == labels).sum().item()

    # count how many samples we've processed so far
    total += imgs.size(0)

# compute average loss and accuracy across the entire training set
train_loss = running_loss / total
train_acc  = correct / total

# TODO: Please compute other metrics and print them

print(f"Train — loss: {train_loss:.3f}, acc: {train_acc:.3f}")

Train — loss: 0.139, acc: 0.947


## Evaluate on the test split

In [48]:
model.eval()               # put the model in *evaluation mode* (disables dropout, uses running stats for batchnorm)
total, correct = 0, 0      # counters to compute accuracy across the whole test set

with torch.no_grad():      # turn off gradient tracking → faster and uses less memory (we're not training)
    for imgs, labels in test_loader:                 # iterate over mini-batches from the test split
        imgs, labels = imgs.to(device), labels.to(device)  # move data to GPU if available (else CPU)

        logits = model(imgs)        # forward pass only (no backward): get raw class scores
        preds = logits.argmax(1)    # pick the class index with the highest score for each image

        correct += (preds == labels).sum().item()  # count how many predictions were correct in this batch
        total   += imgs.size(0)                    # update how many test samples we've seen

# final test accuracy = (# correct predictions) / (total test samples)
test_acc = correct / total

# TODO: Please print metrics in a table and in graphs making the classification performance of the models visible  


## Test the model for single images

In [49]:
class_names = getattr(full_ds, "classes", ["Cat", "Dog"])

def predict_image(img_path):
    """
    Load ONE image, apply the same preprocessing as training,
    run the model, and print Cat/Dog with confidence.
    """
    # 1) Open image and ensure 3-channel RGB
    img = Image.open(img_path).convert("RGB")

    # 2) Preprocess exactly like during training
    x = preprocess(img).unsqueeze(0).to(device)  # shape [1,3,H,W]

    # 3) Inference (no gradients)
    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1)
        conf, pred_idx = probs.max(dim=1)

    # 4) Human-readable output
    pred_label = class_names[int(pred_idx)]
    print(f"Prediction: {pred_label}  (confidence: {conf.item():.1%})")

In [53]:
# Example use (end the path below with the folder and image you prefer):
predict_image(DATA_DIR+"/bacc.jpg")

Prediction: Dog  (confidence: 78.1%)
